In [ ]:
SCAN_SET_ID = "ksr216_2026apr_longC_01"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")

In [ ]:
from pprint import pprint

from rsoxs_unified_workup.config import load_scan_set
from rsoxs_unified_workup.loading import (
    build_base_description,
    scan_ids_by_polarization,
)

configuration = load_scan_set(SCAN_SET_ID)

print(build_base_description(configuration["sample"]))
pprint(scan_ids_by_polarization(configuration))

In [ ]:
import httpx
import PyHyperScattering as phs
from tiled.client import from_uri

catalog = from_uri(
    "https://tiled.nsls2.bnl.gov",
    timeout=httpx.Timeout(None),
)

loader = phs.load.SST1RSoXSDB(
    corr_mode="none",
    catalog=catalog["rsoxs"]["raw"],
)

In [ ]:
from rsoxs_unified_workup.loading import load_scans

scan = load_scans(loader, configuration)

for polarization, scan_data in scan.items():
    print(
        f"Scan {scan_data['scan_id']}: "
        f"configured={polarization}°, "
        f"measured={scan_data['data'].attrs['polarization']}°"
    )

In [ ]:
#at this point, we've made a dict that holds info about each sample and scan conditions
print(type(scan))
print(scan.keys())
print(type(scan[0]))
print(scan[0].keys())
print(scan[0]["sample_name"])
print(scan[0]["data"].shape)
print(scan[0]["data"].dims)

In [ ]:
from importlib import reload

from rsoxs_unified_workup.config import load_scan_set
import rsoxs_unified_workup.plotting as workup_plotting

configuration = load_scan_set(SCAN_SET_ID)
reload(workup_plotting)

if any(
    record["scan_set_id"] != SCAN_SET_ID
    for record in scan.values()
):
    raise ValueError(
        "Run the scan-loading cell for the selected scan set first."
    )

workup_plotting.plot_raw_frames(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
from rsoxs_unified_workup.darks import load_dark_frames

scan = load_dark_frames(loader, scan)

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_dark_histograms,
)

dark_upper_percentile = configuration[
    "processing"
]["darks"]["upper_percentile"]

dark_cutoffs = plot_dark_histograms(
    scan,
    upper_percentile=dark_upper_percentile,
)

{
    polarization: round(float(cutoff), 2)
    for polarization, cutoff in dark_cutoffs.items()
}

In [ ]:
from rsoxs_unified_workup.darks import build_dark_masks

dark_masks, dark_mask_summaries = build_dark_masks(
    scan,
    dark_cutoffs,
)

for polarization, summary in dark_mask_summaries.items():
    print(
        f"Scan {summary['scan_id']}, "
        f"polarization={polarization}°: "
        f"cutoff={summary['cutoff']:.2f}, "
        f"masked pixels={summary['masked_pixels']:,} "
        f"({summary['masked_percent']:.4f}%)"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_dark_masks

plot_dark_masks(scan, dark_masks)

In [ ]:
from rsoxs_unified_workup.masks import (
    combine_static_masks,
    create_beamstop_mask,
    create_border_mask,
)

processing = configuration["processing"]
detector_shape = next(
    iter(scan.values())
)["data"].shape[-2:]

beamstop_mask, beamstop_vertices = create_beamstop_mask(
    detector_shape,
    processing["beamstop"],
)

border_mask = create_border_mask(
    detector_shape,
    processing["detector_borders"],
)

static_masks = combine_static_masks(
    dark_masks,
    beamstop_mask,
    border_mask,
    union_across_polarizations=processing[
        "darks"
    ]["union_across_polarizations"],
)

print("Beamstop vertices:")
for x, y in beamstop_vertices:
    print(f"  ({x:.2f}, {y:.2f})")

print("\nTotal masked pixels:")
for polarization, mask in static_masks.items():
    print(
        f"  polarization={polarization}°: "
        f"{int(mask.sum().item()):,}"
    )

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_static_mask_overlays,
)

plot_static_mask_overlays(
    scan,
    static_masks,
    beamstop_vertices,
)


In [ ]:
from rsoxs_unified_workup.masks import (
    apply_static_masks,
    build_saturation_masks,
)

scan = apply_static_masks(scan, static_masks)
saturation_masks = build_saturation_masks(scan)

for index, energy in enumerate(
    saturation_masks.energy.values
):
    mask = saturation_masks.isel(system=index)
    print(
        f"Energy={energy:.1f} eV: "
        f"{int(mask.sum().item()):,} "
        f"saturated pixels across all polarizations"
    )

In [ ]:
print(type(scan))
print(scan.keys())
print(type(scan[0]))
print(scan[0].keys())
print(scan[0]["sample_name"])

#so we've just added 'static_mask', 'masked_data_static' to the scan dict

In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_saturation_overlays(
    scans=scan,
    saturation_masks=saturation_masks,
    processing=configuration["processing"],
)

In [ ]:
from importlib import reload

import rsoxs_unified_workup.config as workup_config
import rsoxs_unified_workup.masks as workup_masks

reload(workup_config)
reload(workup_masks)

configuration = workup_config.load_scan_set(SCAN_SET_ID)
processing = configuration["processing"]

center_masks, center_mask_summaries = (
    workup_masks.build_center_masks(
        saturation_masks,
        beamstop_vertices,
        processing,
    )
)

for summary in center_mask_summaries:
    print(
        f"Energy={summary['energy']:.1f} eV: "
        f"radius={summary['radius']:.2f} pixels, "
        f"source={summary['radius_source']}, "
        f"farthest point="
        f"({summary['farthest_x']:.2f}, "
        f"{summary['farthest_y']:.2f})"
    )


In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_center_mask_review(
    scans=scan,
    saturation_masks=saturation_masks,
    center_masks=center_masks,
    summaries=center_mask_summaries,
    processing=configuration["processing"],
)

In [ ]:
from rsoxs_unified_workup.masks import (
    apply_final_masks,
    build_final_masks,
)

final_masks = build_final_masks(
    static_masks,
    saturation_masks,
    center_masks,
)

scan = apply_final_masks(scan, final_masks)

for polarization, final_mask in final_masks.items():
    mask_2d = final_mask.isel(system=0).transpose(
        "pix_y",
        "pix_x",
    )

    print(
        f"Polarization={polarization}°: "
        f"full={final_mask.dims} {final_mask.shape}; "
        f"energy 0={mask_2d.dims} {mask_2d.shape}"
    )

In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_final_masked_frames(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
from importlib import reload

import rsoxs_unified_workup.beam_center as beam_center

reload(beam_center)

median_center_suggestion = beam_center.fit_scan_set_beam_centers(
    scans=scan,
    processing=configuration["processing"],
)

for polarization, scan_record in scan.items():
    result = scan_record["beam_center_fit"]
    center = result["fine_best"]

    print(
        f"{polarization}°: "
        f"({center['x']}, {center['y']}), "
        f"cost={center['cost']:.6e}; "
        f"recorded passes={len(result['history'])}"
    )

print("Median suggestion:", median_center_suggestion)

In [ ]:
import rsoxs_unified_workup.plotting as workup_plotting

for scan_record in scan.values():
    workup_plotting.plot_beam_center_fit(
        scan_record["beam_center_fit"],
        scan_record,
    )

In [ ]:
configuration = workup_config.load_scan_set(SCAN_SET_ID)

center_review = beam_center.check_beam_center_consistency(
    scans=scan,
    processing=configuration["processing"],
)

print("Center used for mask:", center_review["center"])
print("Fitted median:", center_review["median"])
print("Difference, pixels:", center_review["difference"])
print(
    "Within fine-step tolerance:",
    center_review["within_tolerance"],
)

The center under [beam_center] in the processing TOML is used
for both the center masks and the starting point of the fit.

Review the fit figures and the consistency check. If the fitted
median differs beyond the tolerance, update [beam_center],
rebuild the center and final masks, and refit.

Once satisfied, explicitly approve that same configured center
for integration using the following cell.

In [ ]:
configuration = workup_config.load_scan_set(SCAN_SET_ID)

center_review = beam_center.check_beam_center_consistency(
    scans=scan,
    processing=configuration["processing"],
)

print("Configured center:", center_review["center"])
print("Fitted median:", center_review["median"])

if not center_review["within_tolerance"]:
    raise ValueError(
        "Update [beam_center] in the processing TOML, "
        "then rebuild the center and final masks and refit."
    )

answer = input(
    "After reviewing the fits, type ACCEPT "
    "to approve this center for integration: "
)

approved_beam_center = beam_center.approve_beam_center(
    scans=scan,
    processing=configuration["processing"],
    approve=(answer.strip() == "ACCEPT"),
)

print("Approved beam center:", approved_beam_center)

In [ ]:
from importlib import reload
import rsoxs_unified_workup.integration as integration

reload(integration)

for polarization, scan_record in scan.items():
    scan_record["integration"] = integration.integrate_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

    clean = scan_record["integration"]["clean"]
    print(
        f"{polarization}°: "
        f"dims={clean.dims}, shape={clean.shape}"
    )

In [ ]:
from importlib import reload

import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

for scan_record in scan.values():
    workup_plotting.plot_integration_review(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

In [ ]:
from importlib import reload
import rsoxs_unified_workup.integration as integration

reload(integration)

for polarization, scan_record in scan.items():
    scan_record["dark_integration"] = integration.integrate_dark_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )

    result = scan_record["dark_integration"]
    print(
        f"{polarization}°: "
        f"dark frame={result['dark_frame_index']}; "
        f"shape={result['darks_clean'].shape}"
    )

In [ ]:
# Integrated darks are stored inside each scan record.

print(type(scan[0]["dark_integration"]))
print(scan[0]["dark_integration"].keys())
print(scan[0]["dark_integration"]["darks_clean"].dims)
print(scan[0]["dark_integration"]["darks_clean"].shape)

In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

for scan_record in scan.values():
    workup_plotting.plot_dark_integration_review(
        scan_record=scan_record,
        processing=configuration["processing"],
    )


In [ ]:
import rsoxs_unified_workup.normalization as normalization

for polarization, scan_record in scan.items():
    scan_record["normalization"] = normalization.normalize_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )
    result = scan_record["normalization"]

    full_check = result[
        "full_ring_normalized"
    ].mean("chi", skipna=True)

    half_region = normalization.select_chi_region(
        result["half_ring_normalized"],
        center_degrees=result[
            "half_ring_center_degrees"
        ],
        halfwidth_degrees=result[
            "half_ring_halfwidth_degrees"
        ],
    )
    half_check = half_region.mean(
        "chi",
        skipna=True,
    )

    full_deviation = np.abs(full_check - 1).max(
        skipna=True
    ).item()
    half_deviation = np.abs(half_check - 1).max(
        skipna=True
    ).item()

    print(
        f"{polarization}°: "
        f"full-ring maximum deviation={full_deviation:.3e}; "
        f"half-ring maximum deviation={half_deviation:.3e}"
    )


In [ ]:
print(type(scan))
print(scan.keys())
print(type(scan[0]["normalization"]))
print(scan[0]["normalization"].keys())
print(scan[0]["normalization"]["full_ring_mean"])

In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_normalization_review(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
import rsoxs_unified_workup.normalization as normalization
import rsoxs_unified_workup.plotting as workup_plotting

for polarization, scan_record in scan.items():
    scan_record["flux_correction"] = normalization.flux_correct_scan(
        scan_record=scan_record,
        processing=configuration["processing"],
    )
    result = scan_record["flux_correction"]

    if not result:
        print(f"{polarization}°: flux correction disabled")
        continue

    flux = result["flux"]
    review_indices = workup_plotting.review_energy_indices(
        scan_record,
        configuration["processing"],
        flux.energy.values,
    )
    review_flux = flux.isel(energy=review_indices)

    print(
        f"{polarization}°: "
        f"corrected shape={result['flux_corrected'].shape}; "
        f"review energies={review_flux.energy.values}; "
        f"Au mesh current={review_flux.values}"
    )

    if review_flux.sizes["energy"] >= 2:
        energy_1, energy_2 = review_flux.energy.values[:2]
        ratio = review_flux.values[0] / review_flux.values[1]

        print(
            f"  Flux ratio ({energy_1:g}/{energy_2:g} eV)"
            f"={ratio:.6f}"
        )

In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_flux_correction_review(
    scans=scan,
    processing=configuration["processing"],
)

In [ ]:
import rsoxs_unified_workup.anisotropy as anisotropy

anisotropy_results = anisotropy.calculate_anisotropy(
    configuration=configuration,
    scans=scan,
)

for sector in anisotropy_results["full_ring"].values():
    values = sector["anisotropy"]
    print(
        f"{sector['label']}: "
        f"parallel={sector['parallel_polarization']}°, "
        f"perpendicular="
        f"{sector['perpendicular_polarization']}°; "
        f"shape={values.shape}; "
        f"finite={np.isfinite(values).sum().item()}"
    )

In [ ]:
workup_plotting.plot_anisotropy_review(
    scans=scan,
    anisotropy_results=anisotropy_results,
)

In [ ]:
workup_plotting.plot_anisotropy_sector_overlays(
    scans=scan,
    configuration=configuration,
    anisotropy_results=anisotropy_results,
    energy_value=configuration["processing"][
        "beam_center_fit"
    ]["fit_energy"],
)

In [ ]:
aggregate_anisotropy_results = (
    anisotropy.aggregate_anisotropy(
        anisotropy_results
    )
)

for mode, results in aggregate_anisotropy_results.items():
    print(f"\n{mode}")

    for method_name in [
        "method_1_global_intensity_ratio",
        "method_2_pointwise_anisotropy",
        "method_3_sector_anisotropy_mean",
    ]:
        values = results[method_name]

        print(
            f"  {method_name}: "
            f"dims={values.dims}, "
            f"shape={values.shape}, "
            f"finite={np.isfinite(values.values).sum()}"
        )

In [ ]:
workup_plotting.plot_aggregate_anisotropy_comparison(
    scans=scan,
    aggregate_results=aggregate_anisotropy_results,
)

In [ ]:
pointwise_anisotropy_results = (
    anisotropy.build_pointwise_anisotropy_maps(
        anisotropy_results
    )
)

for mode, pair_results in (
    pointwise_anisotropy_results.items()
):
    print(f"\n{mode}")

    for pair_id, result in pair_results.items():
        values = result["anisotropy"]

        print(
            f"  {pair_id}: "
            f"dims={values.dims}, "
            f"shape={values.shape}, "
            f"chi range="
            f"{float(values.chi.min()):g} to "
            f"{float(values.chi.max()):g}; "
            f"finite="
            f"{np.isfinite(values.values).sum()}"
        )


In [ ]:
from importlib import reload
import rsoxs_unified_workup.plotting as workup_plotting

reload(workup_plotting)

workup_plotting.plot_pointwise_anisotropy_review(
    scans=scan,
    pointwise_results=pointwise_anisotropy_results,
    processing=configuration["processing"],
)

In [ ]:
from datetime import datetime, timezone
from importlib import reload

import rsoxs_unified_workup.outputs as outputs

reload(outputs)

run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

save_summary = outputs.save_workup_results(
    configuration=configuration,
    scans=scan,
    anisotropy_results=anisotropy_results,
    aggregate_anisotropy_results=aggregate_anisotropy_results,
    pointwise_anisotropy_results=pointwise_anisotropy_results,
    output_root=f"results/runs/{run_id}",
    overwrite=False,
)

for key, value in save_summary.items():
    print(f"{key}: {value}")

In [ ]:
from importlib import reload

import rsoxs_unified_workup.outputs as outputs

reload(outputs)

outputs.print_workup_export_tree(save_summary)

In [ ]:
#notes to self (ignore these for now)

#eventually could make it so that masked individual pixels for union mask only apply for perp and para pairs of scans, i.e., only 0 annd 90 are a union, only 45 and 135 are a union
#could do the same with the center point, where 0 and 90 need to share a center, and 45 and 135 need to share a center, but all four don't need to be shared
#need to account for the fact that the value used for center mask is a guess at center, but is used to draw center radius. ideally would want to find real center before doing center mask. maybe some sort of way to update the center mask and reintegrate again after finding the fitted center
#regroup dictionaries
#probably want to thin kmore deeply about methods 1,2,3 and piontwise exports. what want to export? maybe just do pointwise, and everything beyond that is derived.
#q cap stuff: probably want to log qcap, but we don't stricly need to apply it so early on (before integration)... maybe think about this some more... don't want to throw away good high q data that pointwise anisotropy can catch
#include intensity visualizations (adn save them? or is remeshed enough? are we saving remesed right now? i thin kyes, it's "clean")
#we dont' need to visualize figures for every single energy.
#also, think about how to do all of this stuff when we have a bunch of energys (long_C, long_O, etc)
#include integrated darks in final dictionaries
#do output smarter. zarr, just plain xarray output, etc
#account for some sort of raw images, maybe? thikning mostly of before dark subtraction. could be helpful for stuff when we haven't run enough darks
#half_ring_center_degrees and half_ring_halfwidth_degrees are usuelss i think. what could be mnore useful is knowing the min and max chi used for normalziation calculation
#decide smartly how to use flux when diong anisotropy calcs
#decide smartly how to do concatenation in a way that doesn't introduce chi mismatch
#make figures as an html file
#add commentary
#make some fields into user args (rather than buried in functions): 
    #beamstop vertices, 
    #peak q location for beam centering,
    #which indices to plot
#check on empty (nan) vals in some matrix outputs in check statements
#probably pretty pointless to do whole ring normalization at all, right? actually not sure.
#make it so that integration progress is shown with progress bar rather than with  print statement at each pol/energy
#do pointwise not only for q_capped, but for whole thing
#display the dictionary structure dynamically, maybe during export
#crazy, but is there a way to fit the energy-dependent beam center and fit each energy slice individually, to account for moving beam center?
#make so works with just two pols, or four pols, repeats or longC, flexibility in general
#make all toml set up the same. also understandhow it works.
#add markdown commentary/explanations

#ticket add: switch to xray loading library from caitlyn/priyanka

#make some old legacy functions so that we can dmeonstrate how much better stuff has gotten via a timelapse